In [1]:
# Tải unsloth
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm==0.8.5.post1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.2/47.2 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.4/326.4 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.5/297.5 kB 24.4 MB/s eta 0:00:00


In [2]:
#@title Colab Extra Install { display-mode: "form" }
# Tải các thư viện cần thiết
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm==0.8.5.post1
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.8/162.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 5.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Lin

In [3]:
from unsloth import FastModel, FastLanguageModel,UnslothTrainer, UnslothTrainingArguments
import torch
print("200 OK")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-07-11 14:36:44.016237: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752244604.208287      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752244604.262598      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!
INFO 07-11 14:37:04 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 07-11 14:37:04 [__init__.py:239] Automatically detected platform cuda.
200 OK


In [4]:
# Thư viện cho LLM
from transformers import (
    AutoModelForCausalLM, # Tinh chỉnh mô hình trả lời nhân quả
    AutoTokenizer, # Token hóa câu văn
    BitsAndBytesConfig, # điều chỉnh hiệu quả đông bộ nhớ
    HfArgumentParser,
    TrainingArguments, # Quản lý tham số,
    pipeline,
    logging # ghi lại thông tin loss, accuracy
)

# Tinh chỉnh adapter
from peft import (
    LoraConfig, # Cấu hình cho lora
    PeftModel, # Lưu trữ pp fine tune hiệu quả
    prepare_model_for_kbit_training, # model để training
    get_peft_model # lấy ra sau khi fineTune
)

import os, torch, wandb # wandb giúp lưu trư các thông tin trong quá trình train và vẽ biểu đồ
from datasets import load_dataset
from trl import SFTTrainer, setup_chat_format # Thực hiện training bằng supervised
import bitsandbytes as bnb
print("200 OK")

200 OK


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("PhongGoldFish/aplaca_final_model", device_map=device)
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-3B")
tokenizer.chat_template ="""
{% for message in messages %}
{% if message['role'] == 'system' %}
<|start_of_turn|><|system|>
{{ message['content'] }}<|end_of_turn|>
{% elif message['role'] == 'user' %}
<|start_of_turn|><|user|>
{{ message['content'] }}<|end_of_turn|>
{% elif message['role'] == 'assistant' %}
<|start_of_turn|><|assistant|>
{{ message['content'] }}<|end_of_turn|>
{% endif %}
{% endfor %}
""".strip()
print("200 Ok")

config.json:   0%|          | 0.00/922 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

200 Ok


In [6]:
example = {"instruction":"Nếu bạn là bác sĩ, vui lòng trả lời các câu hỏi y khoa dựa trên mô tả của bệnh nhân.",
           "input":"vi: Bác sĩ, chúng tôi đang lên kế hoạch sinh con và chồng tôi bị thương hàn, kháng sinh được đưa ra trong 3 ngày sốt giảm sau 3 ngày và chúng tôi đã làm thêm xét nghiệm máu, báo cáo cho thấy anh ấy vẫn bị nhiễm nhưng không có dấu hiệu sốt chúng ta nên đợi bao lâu để thụ thai? cảm ơn.",
           "output":""
          }

In [7]:
### có batch

system_prompt = """Bạn là một trợ lý ảo và tư vấn viên chuyên nghiệp cho người Việt Nam.
Nhiệm vụ của bạn là trả lời tất cả câu hỏi liên quan đến kiến thức chung (khoa học, xã hội, đời sống, giáo dục, v.v.) bằng tiếng Việt có thể dùng thuật ngữ tiếng Anh.
Bạn cần sử dụng ngôn ngữ dễ hiểu, văn phong lịch sự, gần gũi, và luôn thể hiện sự tận tình, chu đáo trong từng câu trả lời."""

def format_data_for_sft(example, tokenizer, system_prompt):
    messages = []

    # 1. Thêm tin nhắn của người dùng
    user_message_content = ""
    if example.get('input') and example['input'].strip():
        user_message_content = f"{system_prompt}\n\nCâu hỏi: {example['instruction']}\nThông tin thêm: {example['input']}"
    else:
        user_message_content = f"{system_prompt}\n\nCâu hỏi: {example['instruction']}"

    messages.append({"role": "user", "content": user_message_content})

    # 2. Thêm câu trả lời của trợ lý ảo (model)
    messages.append({"role": "assistant", "content": example['output']})

    # 3. Sử dụng hàm apply_chat_template của tokenizer để tạo chuỗi văn bản cuối cùng.
    #    Hàm này sẽ tự động thêm các token đặc biệt như <start_of_turn>, v.v.
    #    tokenize=False: chỉ trả về string, không trả về input_ids.
    #    add_generation_prompt=False: không thêm prompt để model sinh tiếp, vì ta có cả câu trả lời.
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # 4. Trả về một dictionary với một cột duy nhất là "text"
    return { "text": formatted_text }

formatted_train_data = format_data_for_sft(example,tokenizer,system_prompt)
# In ra một ví dụ để kiểm tra
print("Ví dụ dữ liệu sau khi định dạng:")
print(formatted_train_data['text'])


Ví dụ dữ liệu sau khi định dạng:
<|start_of_turn|><|user|>
Bạn là một trợ lý ảo và tư vấn viên chuyên nghiệp cho người Việt Nam.
Nhiệm vụ của bạn là trả lời tất cả câu hỏi liên quan đến kiến thức chung (khoa học, xã hội, đời sống, giáo dục, v.v.) bằng tiếng Việt có thể dùng thuật ngữ tiếng Anh.
Bạn cần sử dụng ngôn ngữ dễ hiểu, văn phong lịch sự, gần gũi, và luôn thể hiện sự tận tình, chu đáo trong từng câu trả lời.

Câu hỏi: Nếu bạn là bác sĩ, vui lòng trả lời các câu hỏi y khoa dựa trên mô tả của bệnh nhân.
Thông tin thêm: vi: Bác sĩ, chúng tôi đang lên kế hoạch sinh con và chồng tôi bị thương hàn, kháng sinh được đưa ra trong 3 ngày sốt giảm sau 3 ngày và chúng tôi đã làm thêm xét nghiệm máu, báo cáo cho thấy anh ấy vẫn bị nhiễm nhưng không có dấu hiệu sốt chúng ta nên đợi bao lâu để thụ thai? cảm ơn.<|end_of_turn|>
<|start_of_turn|><|assistant|>
<|end_of_turn|>



In [8]:
text = formatted_train_data['text']
inputs = tokenizer(f"{text}", return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=100,
                        repetition_penalty=1.2,# Hạn chế lặp
                        no_repeat_ngram_size=5)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

<|start_of_turn|><|user|>
Bạn là một trợ lý ảo và tư vấn viên chuyên nghiệp cho người Việt Nam.
Nhiệm vụ của bạn là trả lời tất cả câu hỏi liên quan đến kiến thức chung (khoa học, xã hội, đời sống, giáo dục, v.v.) bằng tiếng Việt có thể dùng thuật ngữ tiếng Anh.
Bạn cần sử dụng ngôn ngữ dễ hiểu, văn phong lịch sự, gần gũi, và luôn thể hiện sự tận tình, chu đáo trong từng câu trả lời.

Câu hỏi: Nếu bạn là bác sĩ, vui lòng trả lời các câu hỏi y khoa dựa trên mô tả của bệnh nhân.
Thông tin thêm: vi: Bác sĩ, chúng tôi đang lên kế hoạch sinh con và chồng tôi bị thương hàn, kháng sinh được đưa ra trong 3 ngày sốt giảm sau 3 ngày và chúng tôi đã làm thêm xét nghiệm máu, báo cáo cho thấy anh ấy vẫn bị nhiễm nhưng không có dấu hiệu sốt chúng ta nên đợi bao lâu để thụ thai? cảm ơn.<|end_of_turn|>
<|start_of_turn|><|assistant|>
<|end_of_turn|>
Nếu kết quả xét nghiệm âm tính thì ông sẽ khỏe hơn so với trước đây. Ông nên chờ ít nhất ba tháng nữa mới bắt đầu thụ tinh hợp tử vì điều này cũng giúp tăn